In [1]:
import instructor
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import List
import time

In [2]:

# Define the structure you want
class Step(BaseModel):
    """Describes a step in the AR Training Experience"""
    
    name: str = Field(
        description="A short, descriptive identifier for the step"
    )
    title: str = Field(
        description="A clear and concise task description, to be displayed"
    )
    description: str = Field(
        description="Detailed verbal instructions for the user, guiding them through the task to be performed"
    )

class ARLab(BaseModel):
    """A list of AR training steps"""
    steps: List[Step] = Field(
        description="Complete list of all the steps in the AR training experience"
    )


In [3]:
# Load the environment file with our api keys set
# create a .env file in the same folder, put your api key environment variables there
# Don't forget to put the .env in the gitignore so you don't publish your api keys!
load_dotenv()

# Connect to the LLM with Instructor
client = instructor.from_provider("openai/gpt-5-nano")

# Initialize the tracker for automatic token and retry monitoring
from tracking import InstructorTracker

tracker = InstructorTracker(
    model="gpt-5-nano",
    provider="openai",
    pricing_config={"input": 0.05, "output": 0.40}  # Per 1M tokens
)

# Wrap the client for automatic tracking
tracked_client = tracker.wrap(client)

In [4]:
prompt = """You are an AI agent tasked with generating step by step instructional scenarios for XR training experiences.
Each step is made of the following elements:
Name, Title, and Description.


Generate a training scenario by creating a sequence of steps to accomplish the objective:
Learn how to use the basic functions of a Carvey CNC machine to be able to use it for general operation.
The target audience is:
Students and staff at a university that have varying levels of familiarity with this technology.
The Activity that will be done to accomplish the objective is:
Use the Carvey to carve a stylized letter from wood.

Ensure each step is relevant, accurate, and practical for achieving the training objective within an XR environment.
"""

# Use the tracked client - timing, tokens, retries all captured automatically
lab, completion = tracked_client.create_with_completion(
    response_model=ARLab,
    messages=[
        {"role": "user", "content": prompt}
    ]
)

In [5]:
# Get metrics from the tracker - includes retries that were previously invisible!
metrics = tracker.get_last_call_metrics()

print("=== Per-Call Metrics ===")
print(f"Total Attempts: {metrics.total_attempts}")
print(f"Retries: {metrics.retry_count}")
print(f"Parse Errors: {metrics.parse_errors}")
print(f"API Errors: {metrics.api_errors}")
print(f"Duration: {metrics.duration_ms:.2f}ms")
print()

# Token usage now includes ALL tokens from retries
print("=== Token Usage (including retries) ===")
print(f"Prompt tokens:     {metrics.total_usage.prompt_tokens:,}")
print(f"Completion tokens: {metrics.total_usage.completion_tokens:,}")
print(f"Total tokens:      {metrics.total_usage.total_tokens:,}")
print()

# Cost is calculated automatically
print(f"=== Cost ===")
print(f"Total Cost: ${tracker.get_call_cost(metrics.call_id):.6f}")

=== Per-Call Metrics ===
Total Attempts: 1
Retries: 0
Parse Errors: 0
API Errors: 0
Duration: 32374.08ms

=== Token Usage (including retries) ===
Prompt tokens:     353
Completion tokens: 3,975
Total tokens:      4,328

=== Cost ===
Total Cost: $0.001608


In [6]:
# Print the generated steps
print("=== Generated AR Training Steps ===\n")
for i, step in enumerate(lab.steps):
    print(f"Step {i}: {step.name}")
    print(f"  Title: {step.title}")
    print(f"  Description: {step.description}\n")

=== Generated AR Training Steps ===

Step 0: intro_safety
  Title: Safety Orientation and Objective
  Description: In this XR training module, you will learn the basic Carvey CNC operations through a safe, guided scenario. Begin with personal protective equipment (PPE) reminders, hazards to avoid (rotating spindle, sharp bits, moving gantry), and the objective: use basic Carvey functions to carve a stylized letter from wood. You’ll progress from setup to toolpath execution and finishing the carved piece.

Step 1: carvey_overview
  Title: Carvey Essentials: Components and Axes
  Description: Explore the Carvey’s main components in the XR environment: spindle and collet, wasteboard, clamps, dust collection, and the control panel. Understand the three Cartesian axes (X, Y, Z) and how tool paths translate into machine motions. Identify emergency stop, Power switch, and screen controls.

Step 2: workspace_setup
  Title: Prepare Workspace and Material
  Description: Check your virtual wood b

In [7]:
# Print full tracking summary
tracker.print_summary()


TRACKING SUMMARY - gpt-5-nano (openai)
Session ID: c39be5fa

Calls: 1 total, 1 success, 0 failed (100.0%)

Tokens:
  Prompt:     353
  Completion: 3,975
  Total:      4,328

Retries:
  Total retries: 0
  Calls needing retries: 0 (0.0%)
  Avg retries/call: 0.00
  Max retries (single call): 0

Errors:
  Parse errors: 0
  API errors: 0
  Completion errors: 0

Timing:
  Total: 32374.1ms
  Average: 32374.1ms per call

Cost: $0.001608



In [8]:
# Export for benchmarking (optional)
# from tracking import BenchmarkExporter
# 
# # Export to JSON
# BenchmarkExporter.to_json(tracker.aggregate, "benchmark_gpt5nano.json")
# 
# # Or get markdown report
# report = BenchmarkExporter.to_markdown_report(tracker.aggregate, tracker.pricing)
# print(report)